# 09 · Gradient precision and speed across FlagQuantum runtimes

Compare the same circuit, parameters, observable, and PyTorch gradient loop across statevector, MPS, and tensor-network policies. These small CPU measurements are development evidence, not scalability claims.

In [ ]:
import time
import torch
import flagquantum as fq

N_QUBITS = 4
INITIAL = torch.linspace(-0.3, 0.3, steps=N_QUBITS)

def ansatz(parameters):
    circuit = fq.Circuit(n_qubits=N_QUBITS)
    for qubit in range(N_QUBITS):
        circuit.ry(qubit=qubit, theta=parameters[qubit])
    for qubit in range(N_QUBITS - 1):
        circuit.cx(control=qubit, target=qubit + 1)
    return circuit

In [ ]:
def value_gradient_and_time(mode):
    model = fq.Module(
        ansatz,
        n_parameters=N_QUBITS,
        init=INITIAL,
        policy=fq.RuntimePolicy(
            execution_options=fq.ExecutionOptions(mode=mode),
            observable='z_sum',
            observable_wires=tuple(range(N_QUBITS)),
        ),
    )
    started = time.perf_counter()
    value = model()
    value.sum().backward()
    elapsed = time.perf_counter() - started
    return {
        'mode': mode,
        'value': float(value.detach()),
        'gradient': model.parameters_tensor.grad.detach().clone(),
        'seconds': elapsed,
        'distribution_semantics': model.execute().runtime['distribution_semantics'],
    }

rows = [value_gradient_and_time(mode) for mode in ('statevector', 'mps', 'tensor_network')]

In [ ]:
reference = rows[0]
for row in rows:
    row['value_abs_error'] = abs(row['value'] - reference['value'])
    row['gradient_max_abs_error'] = float((row['gradient'] - reference['gradient']).abs().max())
    printable = {key: value for key, value in row.items() if key != 'gradient'}
    print(printable)
    assert row['distribution_semantics'] == 'single_device_fast_path'
    assert row['value_abs_error'] < 1e-5
    assert row['gradient_max_abs_error'] < 1e-4